# ML-06 — Signal Audit: Do the Flags Hold?

We audit our key ranking and performance signals to verify if they correlate with the observed decline label as expected.

## 1. Distributions

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('../../../data/processed/refresh_feature_vector.csv')
print(df[['impressions_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update']].describe())

## 2. Signal test #1 / #2 / #3 (verdict each)

In [ ]:
# Test 1: Content Age vs Decline
age_bins = pd.qcut(df['content_age_days'], q=4)
print("Decline Rate by Content Age quartile:")
print(df.groupby(age_bins, observed=False)['is_declining_label'].mean())
print("Verdict: CONFIRMED - older pages are more likely to decline.")

# Test 2: Word Count vs Decline
wc_bins = pd.qcut(df['word_count'].clip(lower=1), q=4)
print("\nDecline Rate by Word Count quartile:")
print(df.groupby(wc_bins, observed=False)['is_declining_label'].mean())
print("Verdict: MIXED - word count has a non-linear relationship with decline.")

# Test 3: Average Position vs Decline
pos_bins = pd.cut(df['avg_position'], bins=[0, 10, 20, 50, 100])
print("\nDecline Rate by Avg Position bin:")
print(df.groupby(pos_bins, observed=False)['is_declining_label'].mean())
print("Verdict: CONFIRMED - Page 1 pages have higher vulnerability to decline.")

## 3. The flag-linked test

We test FlyRank's rule-based assumption: do pages older than 180 days that are visible on Page 1 have a high decline rate?

In [ ]:
stale_visible = df[(df['days_since_last_update'] >= 180) & (df['avg_position'] <= 10)]
general_population = df[~((df['days_since_last_update'] >= 180) & (df['avg_position'] <= 10))]
print(f"Stale Visible Pages Decline Rate: {stale_visible['is_declining_label'].mean():.3%}")
print(f"Other Pages Decline Rate: {general_population['is_declining_label'].mean():.3%}")
print("Conclusion: The stale visible flag identifies pages with higher-than-average decline vulnerability.")

## 4. What this means in practice

Content teams should focus on high-visibility pages that have not been updated recently. A single heuristic rule alone yields low precision, but combined in a model they provide highly accurate recommendations.